# 🧬 Sanger Sequencing & Modern Methods

---

## Learning Objectives

- Understand Sanger dideoxy sequencing
- Learn to read sequencing chromatograms
- Compare with next-generation sequencing
- Analyze trace files

---

## Sanger Sequencing Principle

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    SANGER DIDEOXY SEQUENCING                            │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Chain Termination Method (1977, Nobel Prize)                          │
│                                                                         │
│   Normal dNTP:           Dideoxy dNTP (ddNTP):                          │
│        O                        O                                       │
│        ║                        ║                                       │
│    O-P-O-5'CH2  Base        O-P-O-5'CH2  Base                           │
│        ║     \   │              ║     \   │                             │
│        O      O─┘               O      O─┘                              │
│              ╱                        ╱                                 │
│           3'OH  ← Can extend      3'H  ← STOP! No 3'OH                  │
│                                                                         │
│   Reaction mixture:                                                     │
│   - Template DNA                                                        │
│   - Primer                                                              │
│   - DNA polymerase                                                      │
│   - dNTPs (normal, high concentration)                                  │
│   - ddNTPs (terminators, low concentration, fluorescently labeled)      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## Sequencing Products

```
Template:  3'-ATCGATCGATCGATCG-5'
Primer:    5'-TAGC...

Products after chain termination:

                        ddA termination (Green)
   5'-TAGCTAGCTA*                  (10 bases)
   5'-TAGCTAGCTAGCTA*              (14 bases)

                        ddT termination (Red)
   5'-TAGCT*                       (5 bases)
   5'-TAGCTAGCT*                   (9 bases)
   5'-TAGCTAGCTAGCT*               (13 bases)

                        ddG termination (Black)
   5'-TAG*                         (3 bases)
   5'-TAGCTAG*                     (7 bases)
   5'-TAGCTAGCTAG*                 (11 bases)

                        ddC termination (Blue)
   5'-TAGC*                        (4 bases)
   5'-TAGCTAGC*                    (8 bases)
   5'-TAGCTAGCTAGC*                (12 bases)

Capillary electrophoresis separates by size:
┌─────────────────────────────────────────────────────────────────────────┐
│ Size→   3   4   5   6   7   8   9  10  11  12  13  14                   │
│         G   C   T       G   C   T   A   G   C   T   A                   │
│         ▓   █   ▒       ▓   █   ▒   ░   ▓   █   ▒   ░   ← Peaks        │
│                                                                         │
│ Read:   T   A   G   C   T   A   G   C   T   A   G   C                   │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def simulate_sanger_products(template, primer_start, max_length=20):
    """
    Simulate Sanger sequencing products.
    
    Returns dictionary of fragment lengths for each ddNTP.
    """
    # Reverse complement for synthesis direction
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    template = template.upper()[primer_start:]
    
    products = {'A': [], 'T': [], 'G': [], 'C': []}
    
    # Synthesize complement, record termination points
    for i, base in enumerate(template[:max_length]):
        new_base = complement.get(base, 'N')
        products[new_base].append(i + 1)  # 1-indexed length
    
    return products

def reconstruct_sequence(products):
    """
    Reconstruct sequence from termination products.
    """
    # Create position→base mapping
    pos_to_base = {}
    for base, positions in products.items():
        for pos in positions:
            pos_to_base[pos] = base
    
    # Build sequence
    max_pos = max(pos_to_base.keys()) if pos_to_base else 0
    sequence = ''.join(pos_to_base.get(i, 'N') for i in range(1, max_pos + 1))
    
    return sequence

# Example
template = "ATCGATCGATCGATCG"
products = simulate_sanger_products(template, 0)

print(f"Template: 3'-{template}-5'")
print(f"\nTermination positions:")
for base, positions in products.items():
    print(f"  dd{base}: {positions}")

reconstructed = reconstruct_sequence(products)
print(f"\nReconstructed: 5'-{reconstructed}-3'")

---

## Reading Chromatograms

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    CHROMATOGRAM INTERPRETATION                          │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Quality indicators:                                                   │
│                                                                         │
│   GOOD:                         BAD:                                    │
│     ╱╲   ╱╲   ╱╲   ╱╲            ╱╲                                     │
│    ╱  ╲ ╱  ╲ ╱  ╲ ╱  ╲          ╱  ╲╱╲                                  │
│   ╱    ╲    ╲    ╲    ╲        ╱ ╱╲  ╲╱╲  ← Overlapping peaks          │
│   ─────────────────────       ─────────────                             │
│    A    T    G    C             ?   ??                                  │
│                                                                         │
│   Common issues:                                                        │
│   1. Dye blobs at start (unincorporated ddNTPs)                         │
│   2. Signal decay at end (polymerase processivity)                      │
│   3. Double peaks (heterozygosity or contamination)                     │
│   4. Compressions (secondary structure)                                 │
│                                                                         │
│   Quality Score (Phred):                                                │
│   Q = -10 × log₁₀(P_error)                                              │
│                                                                         │
│   Q20 = 99% accuracy (1 error per 100 bases)                            │
│   Q30 = 99.9% accuracy (1 error per 1000 bases)                         │
│   Q40 = 99.99% accuracy (1 error per 10000 bases)                       │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def parse_ab1_quality(ab1_file):
    """
    Parse quality scores from AB1 trace file.
    
    Note: Requires biopython for full implementation.
    This is a simplified demonstration.
    """
    try:
        from Bio import SeqIO
        record = SeqIO.read(ab1_file, "abi")
        
        sequence = str(record.seq)
        qualities = record.letter_annotations.get("phred_quality", [])
        
        return {
            'sequence': sequence,
            'qualities': qualities,
            'avg_quality': sum(qualities) / len(qualities) if qualities else 0
        }
    except ImportError:
        return {'error': 'BioPython required for AB1 parsing'}

def quality_trim(sequence, qualities, min_quality=20, window=10):
    """
    Trim sequence based on quality scores.
    
    Uses sliding window approach to find high-quality region.
    """
    if not qualities:
        return sequence, 0, len(sequence)
    
    # Find start (first window with good quality)
    start = 0
    for i in range(len(qualities) - window):
        window_avg = sum(qualities[i:i+window]) / window
        if window_avg >= min_quality:
            start = i
            break
    
    # Find end (last window with good quality)
    end = len(qualities)
    for i in range(len(qualities) - window, start, -1):
        window_avg = sum(qualities[i:i+window]) / window
        if window_avg >= min_quality:
            end = i + window
            break
    
    return sequence[start:end], start, end

# Example with synthetic data
seq = "NNNATCGATCGATCGATCGATCGATCNN"
qual = [5, 5, 5, 30, 35, 38, 40, 38, 35, 33, 35, 38, 40, 38, 35, 33, 35, 38, 40, 38, 35, 33, 25, 20, 15, 10, 5, 5]

trimmed, start, end = quality_trim(seq, qual, min_quality=25, window=5)
print(f"Original: {seq}")
print(f"Trimmed:  {trimmed}")
print(f"Region: {start}-{end}")

---

## Sanger vs NGS Comparison

```
┌─────────────────────────────────────────────────────────────────────────┐
│                 SEQUENCING METHOD COMPARISON                            │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Feature          │ Sanger        │ Illumina       │ PacBio/ONT       │
│   ─────────────────┼───────────────┼────────────────┼─────────────────  │
│   Read length      │ 700-1000 bp   │ 150-300 bp     │ 10-50+ kb        │
│   Throughput       │ Low           │ Very High      │ Medium           │
│   Accuracy         │ 99.99%        │ 99.9%          │ 85-99%           │
│   Cost per base    │ High          │ Very Low       │ Medium           │
│   Run time         │ 1-3 hours     │ 1-3 days       │ Hours-days       │
│                                                                         │
│   Best for:                                                             │
│   • Sanger: Validation, small targets, mixed samples                    │
│   • Illumina: WGS, RNA-seq, amplicon sequencing                         │
│   • Long-read: Structural variants, de novo assembly, full-length       │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## 🏋️ Exercises

### Exercise 1: Primer Design
Design sequencing primers for a given template.

### Exercise 2: Quality Analysis
Analyze quality scores and determine usable read length.

### Exercise 3: Heterozygote Detection
Identify heterozygous positions from peak ratios.

---

## 📚 Resources

- [FinchTV](https://digitalworldbiology.com/FinchTV) - Free chromatogram viewer
- [4Peaks](https://nucleobytes.com/4peaks/) - Mac chromatogram viewer
- [BioPython AB1 Tutorial](https://biopython.org/wiki/ABI_Traces)